# STAC AI Workshop · NYC · 21 October 2026

## 04 · Adaptation lab: data, learning and blind spots

**Participant workbook · Matthew Dixon, PhD, FRM**

### BeliefLens · STAC AI Workshop · NYC · 21 October 2026

## Learning objectives
1. Prepare financial instruction examples
2. Separate training and evaluation data
3. Train and compare a small model
4. Diagnose a missing feature and generalization limits

All companies, numbers, labels and market observations in these notebooks are synthetic teaching examples. Offline outputs come from explicit deterministic rules, not an LLM. The same JavaScript case engine runs in the browser and through this Python client. Only a cell explicitly using `live=True` calls Together AI. No financial action is executed.

**Requirements:** Python 3.10+, Node.js 20+, Jupyter. No model weights or API key are needed for the default exercises. Run cells in order.

In [ ]:
from pathlib import Path
import sys, json, math, random, statistics
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_client.py").exists()), None)
if root is None:
    raise RuntimeError("Extract the complete workshop ZIP and open Jupyter inside its folder.")
if str(root) not in sys.path: sys.path.insert(0, str(root))
from lab_client import run, compact, save
print("Shared engine ready:", compact(run()))

## Build examples with explicit targets
These records could inform SFT dataset construction, subject to expert review. Labels come from the declared teaching rule, not from another LLM. The numerical exercise trains a logistic classifier, **not a language model**. It exposes why the features and examples available to a learner matter.

In [ ]:
rng = random.Random(42)
rows = []
for i in range(240):
    leverage = rng.uniform(1, 5)
    disruption = int(rng.random() < 0.3)
    label = int(leverage > 3.5 or disruption == 1)
    rows.append({"id":f"case-{i}", "issuer":f"issuer-{i//8}", "leverage":leverage, "disruption":disruption, "review":label})
train = [r for r in rows if int(r["issuer"].split("-")[1]) < 20]
test = [r for r in rows if int(r["issuer"].split("-")[1]) >= 20]
assert not ({r["issuer"] for r in train} & {r["issuer"] for r in test})
sample = train[0]
instruction = {"messages":[{"role":"user","content":f"Assess leverage {sample['leverage']:.2f}x; disruption={sample['disruption']}"},{"role":"assistant","content":"REVIEW" if sample["review"] else "NO EXCEPTION UNDER THIS RULE"}]}
print(json.dumps(instruction, indent=2))
print("Training / held out:", len(train), len(test))

In [ ]:
def features(row, include_disruption):
    return [1., row["leverage"]-3.5] + ([float(row["disruption"])] if include_disruption else [])
def probability(w, x):
    z = max(-30, min(30, sum(a*b for a,b in zip(w,x))))
    return 1/(1+math.exp(-z))
def fit(data, include_disruption):
    w = [0.] * (3 if include_disruption else 2)
    for epoch in range(800):
        gradient = [0.]*len(w)
        for row in data:
            x=features(row,include_disruption); error=probability(w,x)-row["review"]
            for j in range(len(w)): gradient[j]+=error*x[j]/len(data)
        w=[v-0.15*g for v,g in zip(w,gradient)]
    return w
def evaluate(w, data, include_disruption):
    predictions=[int(probability(w,features(r,include_disruption))>=0.5) for r in data]
    misses=sum(p==0 and r["review"]==1 for p,r in zip(predictions,data))
    return {"accuracy":sum(p==r["review"] for p,r in zip(predictions,data))/len(data), "missed_review":misses}
for full in [False,True]:
    w=fit(train,full)
    print("Includes disruption:",full,"train:",evaluate(w,train,full),"test:",evaluate(w,test,full))

## Compare shared-engine configurations
These are controller experiments, separate from the trained classifier. A calculator can improve numerical traceability while the release decision remains blocked by a missing approval.

In [ ]:
for mode in ["model_only","rag","rag_tools"]:
    print(compact(run({"mode":mode,"disruption":False})))

## Learning check
Which is the main lesson of adding the disruption feature? **A:** more parameters always help; **B:** relevant state omitted from inputs can cause systematic misses; **C:** classification accuracy proves governance safety.

**Answer: B.** The split prevents issuer overlap, but these synthetic data still share a simple generating process. Real evaluation needs realistic temporal, issuer and failure-mode shifts.

**Exercise:** create an additional holdout with changed covenant definitions. Which labels or features would need review?

## What you have demonstrated
- **Prepare financial instruction examples:** Constructed source/target examples with declared labels
- **Separate training and evaluation data:** Prevented issuer overlap
- **Train and compare a small model:** Actually fitted two logistic models and measured held-out errors
- **Diagnose a missing feature and generalization limits:** Compared a model with and without a risk-relevant feature

**Exit ticket:** State one result, the evidence supporting it, and one limit that prevents you from claiming more. Save a run JSON or your numerical results as the experiment record.